# 🧠 Neural Networks & Deep Learning
**ISLP Chapter 10 · Data Pattern Recognition for the Rest of Us**

> Neural networks learn hierarchical representations from data. For tabular business data, even a shallow network often outperforms linear models. For images, text, and sequences, deep networks are transformative.

### Dataset
**Employee Compensation Analytics**
Predict annual salary from job title, department, years of experience, education, and performance metrics. A real HR/compensation analytics problem where neural networks can capture nonlinear interactions (e.g., seniority × department effects).

### Time: ~65 minutes

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.facecolor':'white','axes.facecolor':'#f8f9fa',
    'axes.grid':True,'grid.alpha':0.4,'axes.spines.top':False,'axes.spines.right':False,'font.size':11})
from sklearn.neural_network import MLPRegressor, MLPClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.linear_model import LinearRegression
print("\u2713 Setup complete")

In [ ]:
# Employee Compensation Dataset
# Realistic salary data with nonlinear interactions between features
np.random.seed(42)
n = 3000

departments = ['Engineering','Finance','Marketing','Sales','Operations','HR','Legal']
job_levels  = ['Junior','Mid','Senior','Lead','Director','VP']
education   = ['High School','Associate','Bachelor','Master','PhD']

dept = np.random.choice(departments, n, p=[0.30,0.15,0.12,0.18,0.12,0.08,0.05])
level= np.random.choice(job_levels, n, p=[0.20,0.28,0.25,0.15,0.08,0.04])
edu  = np.random.choice(education, n, p=[0.05,0.08,0.45,0.32,0.10])
yoe  = np.random.randint(0, 25, n).astype(float)
perf = np.random.randint(1, 6, n).astype(float)  # 1-5 performance rating
age  = (22 + yoe + np.random.normal(0, 2, n)).clip(22, 65)

# Encode levels
dept_pay  = {'Engineering':1.3,'Finance':1.2,'Legal':1.25,'Marketing':1.0,
             'Sales':0.95,'Operations':0.9,'HR':0.88}
level_mult= {'Junior':0.6,'Mid':0.85,'Senior':1.1,'Lead':1.35,'Director':1.7,'VP':2.2}
edu_bonus = {'High School':0,'Associate':0.03,'Bachelor':0.08,'Master':0.15,'PhD':0.22}

# Nonlinear salary formula: interaction between dept, level, experience
base = 55000
salary = np.array([
    base
    * dept_pay[dept[i]]
    * level_mult[level[i]]
    * (1 + edu_bonus[edu[i]])
    * (1 + 0.025 * min(yoe[i], 15))   # experience premium plateaus
    * (1 + 0.04 * (perf[i] - 3))      # performance premium
    * np.random.lognormal(0, 0.08)     # noise
    for i in range(n)
])

df = pd.DataFrame({
    'Department': dept, 'JobLevel': level, 'Education': edu,
    'YearsExperience': yoe, 'PerformanceRating': perf,
    'Age': age, 'Salary': salary.round(-2)
})

print(f"Employee Compensation dataset: {df.shape}")
print(f"\nSalary range: ${df['Salary'].min():,.0f} — ${df['Salary'].max():,.0f}")
print(f"Median salary: ${df['Salary'].median():,.0f}")
print(f"\nBreakdown by level:")
print(df.groupby('JobLevel')['Salary'].agg(['mean','count']).sort_values('mean').round(0).to_string())

## 📐 Part 1 — From Neuron to Network

**A single neuron computes:**
```
a = f(w₀ + w₁x₁ + w₂x₂ + ... + wₚxₚ)
```
where f is a nonlinear activation function (ReLU: max(0,x)).

**Why layers matter:** Each layer transforms the data into a new representation. Layer 1 might learn "seniority × department" interactions. Layer 2 might learn how those interact with education. The network automatically discovers these combinations from data.

**For tabular data:** Even a shallow network (1-2 hidden layers) often beats linear models by capturing interactions and nonlinearities without explicit feature engineering.

In [ ]:
# Prepare features
le = LabelEncoder()
df_enc = df.copy()
for col in ['Department','JobLevel','Education']:
    df_enc[col] = le.fit_transform(df[col])

X = df_enc.drop('Salary', axis=1).values
y = np.log(df_enc['Salary'].values)  # predict log(salary) — stabilizes variance

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr)
X_te_s  = scaler.transform(X_te)

# Compare: Linear Regression vs Neural Network
lr = LinearRegression().fit(X_tr_s, y_tr)
lr_pred = lr.predict(X_te_s)

nn_shallow = MLPRegressor(hidden_layer_sizes=(64,), activation='relu',
                           max_iter=500, random_state=42, early_stopping=True)
nn_shallow.fit(X_tr_s, y_tr)
nn_s_pred = nn_shallow.predict(X_te_s)

nn_deep = MLPRegressor(hidden_layer_sizes=(128, 64, 32), activation='relu',
                        max_iter=500, random_state=42, early_stopping=True)
nn_deep.fit(X_tr_s, y_tr)
nn_d_pred = nn_deep.predict(X_te_s)

print(f"{'Model':<30} {'R²':>8} {'MAE ($)':>12} {'RMSE ($)':>12}")
print("-" * 65)
for name, pred in [('Linear Regression', lr_pred),
                   ('Shallow NN (64)', nn_s_pred),
                   ('Deep NN (128-64-32)', nn_d_pred)]:
    r2   = r2_score(y_te, pred)
    mae  = mean_absolute_error(np.exp(y_te), np.exp(pred))
    rmse = np.sqrt(mean_squared_error(np.exp(y_te), np.exp(pred)))
    print(f"  {name:<28} {r2:>8.4f} {mae:>12,.0f} {rmse:>12,.0f}")
print("\nMAE/RMSE are in original $ scale (we exponentiate log predictions)")

In [ ]:
# Visualize: actual vs predicted salary + learning curves
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Actual vs predicted
y_pred_dollars = np.exp(nn_deep.predict(X_te_s))
y_actual_dollars = np.exp(y_te)

axes[0].scatter(y_actual_dollars/1000, y_pred_dollars/1000,
                alpha=0.3, s=12, color='#1e5fa8')
max_val = max(y_actual_dollars.max(), y_pred_dollars.max()) / 1000
axes[0].plot([0,max_val],[0,max_val],'k--',lw=1.5,alpha=0.5,label='Perfect prediction')
axes[0].set_xlabel('Actual Salary ($000s)')
axes[0].set_ylabel('Predicted Salary ($000s)')
axes[0].set_title(f'Neural Network: Actual vs Predicted Salary\n'
                  f'R² = {r2_score(y_te, nn_deep.predict(X_te_s)):.3f}')
axes[0].legend()

# Residuals by department
pred_all = np.exp(nn_deep.predict(scaler.transform(X)))
residuals_pct = (pred_all - df['Salary'].values) / df['Salary'].values * 100

for dept_name in departments:
    mask = df['Department'] == dept_name
    axes[1].boxplot(residuals_pct[mask], positions=[departments.index(dept_name)],
                    widths=0.6, patch_artist=True,
                    boxprops=dict(facecolor='#1e5fa8', alpha=0.6))

axes[1].axhline(0, color='#e85d20', lw=2, ls='--')
axes[1].set_xticks(range(len(departments)))
axes[1].set_xticklabels([d[:4] for d in departments], rotation=45)
axes[1].set_ylabel('Prediction Error (%)')
axes[1].set_title('Residuals by Department\n(should be centered at 0 for unbiased model)')

plt.tight_layout(); plt.show()

In [ ]:
# Activation functions — why nonlinearity matters
x_range = np.linspace(-4, 4, 200)
activations = {
    'ReLU': np.maximum(0, x_range),
    'Sigmoid': 1/(1+np.exp(-x_range)),
    'Tanh': np.tanh(x_range),
    'Linear (no activation)': x_range,
}
fig, ax = plt.subplots(figsize=(9, 4))
colors_act = ['#1e5fa8','#e85d20','#1a7a45','#888']
for (name, vals), color in zip(activations.items(), colors_act):
    lw = 2.5 if name == 'ReLU' else 1.5
    ls = '-' if name != 'Linear (no activation)' else '--'
    ax.plot(x_range, vals, label=name, color=color, lw=lw, ls=ls)
ax.axhline(0, color='black', lw=0.5, alpha=0.3)
ax.axvline(0, color='black', lw=0.5, alpha=0.3)
ax.set_xlabel('Input value')
ax.set_ylabel('Output value')
ax.set_title('Activation Functions — why nonlinearity is essential\n'
             'Without it, a 10-layer network reduces to a single linear transformation')
ax.legend()
ax.set_ylim(-1.5, 4)
plt.tight_layout(); plt.show()
print("\n\u2714 ReLU is the default for tabular data: fast, avoids vanishing gradients")
print("   Without nonlinear activations, deep networks are mathematically equivalent")
print("   to a single linear regression — no benefit from the extra layers")

In [ ]:
# @title 📝 Quiz — Neural Networks & Deep Learning
# @markdown Answer each question in the box below, then run the AI grading cell.

# @markdown **Q1:** What does a single neuron compute? Describe the two steps.
# @markdown **Q2:** Why do neural networks need nonlinear activation functions?
# @markdown **Q3:** What is backpropagation and what problem does it solve?
# @markdown **Q4:** On the compensation dataset, the NN outperformed linear regression. Why?
# @markdown **Q5:** Name one advantage and one disadvantage of neural networks vs Random Forests for tabular HR data.

q1 = "" # @param {type:"string", placeholder:"your answer"}
q2 = "" # @param {type:"string", placeholder:"your answer"}
q3 = "" # @param {type:"string", placeholder:"your answer"}
q4 = "" # @param {type:"string", placeholder:"your answer"}
q5 = "" # @param {type:"string", placeholder:"your answer"}

# Collect into answers dict for grading cell
answers = {"q1": q1, "q2": q2, "q3": q3, "q4": q4, "q5": q5}
missing = [k for k,v in answers.items() if not str(v).strip()]
print(f"  {5-len(missing)}/5 answered — run the AI grading cell below for feedback")

### 📤 Submit for AI Grading

In [ ]:
_NB_NAME_ = "Neural Networks & Deep Learning"
# @title 🤖 AI Feedback — enter your username and click ▶ Run
# @markdown **No API key needed.** AI grading runs free inside your Colab session.
# @markdown
# @markdown Enter your GitHub username below so your instructor can track your submission, then click ▶ Run.

GITHUB_USERNAME = "" # @param {type:"string", placeholder:"e.g. jsmith42"}

# ── runs automatically — do not edit below ───────────────────
import json, textwrap, re as _re
_NB_TITLE = globals().get("_NB_NAME_", "this notebook")

def _grade(answers_dict, nb_name):
    answered = {k: v for k, v in answers_dict.items() if str(v).strip()}
    n_total  = len(answers_dict)
    qa       = "\n".join(f"Q{k.replace('q','')}: {v}" for k,v in answered.items())
    prompt   = (f"You are a TA grading quiz answers for \"{nb_name}\" in a data science "
                f"course (Data Pattern Recognition for the Rest of Us, based on ISLP).\n\n"
                f"Student answers ({len(answered)}/{n_total}):\n{qa or '(none)'}\n\n"
                f"Grade conceptual understanding. Accept reasonable paraphrases. "
                f"Be encouraging and specific. Reply ONLY in this JSON (no markdown):\n"
                f"{{\"quiz_score\":<int 0-{n_total}>,"
                f"\"grade\":\"<Excellent|Good|Needs Review|Incomplete>\","
                f"\"feedback\":\"<2-3 sentences>\","
                f"\"tip\":\"<one takeaway>\"}}") 
    try:
        import google.generativeai as genai
        import google.auth, google.auth.transport.requests
        creds, _ = google.auth.default()
        creds.refresh(google.auth.transport.requests.Request())
        genai.configure(credentials=creds)
        r = genai.GenerativeModel("gemini-2.0-flash").generate_content(prompt)
        return r.text, "Gemini via Colab"
    except Exception:
        pass
    try:
        from google.colab import userdata
        key = userdata.get("GEMINI_API_KEY")
        if key and len(key) > 10:
            import google.generativeai as genai
            genai.configure(api_key=key)
            r = genai.GenerativeModel("gemini-2.0-flash").generate_content(prompt)
            return r.text, "Gemini via key"
    except Exception:
        pass
    return None, None

def _fallback(answers_dict):
    n = len(answers_dict)
    done = [v for v in answers_dict.values() if str(v).strip()]
    nd, avg = len(done), sum(len(str(v)) for v in done)/max(len(done),1)
    if nd == 0: return {"quiz_score":0,"grade":"Incomplete","feedback":"No answers — fill in the quiz above.","tip":"Each question needs 1-2 sentences."}
    if avg < 15: return {"quiz_score":nd,"grade":"Needs Review","feedback":f"{nd}/{n} answered but very brief. Explain your reasoning.","tip":"Aim for 1-2 sentences per answer."}
    if nd == n:  return {"quiz_score":n,"grade":"Good","feedback":f"All {n} questions answered.","tip":"Review any concepts that felt unclear."}
    return {"quiz_score":nd,"grade":"Needs Review","feedback":f"{nd}/{n} answered. Complete the remaining {n-nd}.","tip":"Answer all questions for full credit."}

def _show(result, username, nb_name, source):
    C = {"Excellent":"\033[92m","Good":"\033[94m","Needs Review":"\033[93m","Incomplete":"\033[91m"}
    R = "\033[0m"; g = result.get("grade","?")
    n = len(answers); qs = result.get("quiz_score",0)
    print("\n"+"\u2550"*56)
    print(f"  \U0001f916 AI Feedback \u2014 {nb_name}")
    if source: print(f"  Powered by   {source}")
    print("\u2550"*56)
    print(f"  Student : {'@'+username if username else '\u26a0 set GITHUB_USERNAME above'}")
    print(f"  Grade   : {C.get(g,'')} {g} {R}")
    print(f"  Score   : {qs}/{n}  [{'\u2588'*qs+'\u2591'*(n-qs)}]")
    print()
    [print(f"  {l}") for l in textwrap.wrap(result.get("feedback",""),54)]
    tip = result.get("tip","")
    if tip:
        print()
        [print(f"  \U0001f4a1 {l}") for l in textwrap.wrap(tip,52)]
    print("\u2550"*56+"\n")

if "answers" not in globals():
    print("\u26a0  Run the quiz cell above first, then re-run this cell.")
else:
    nd = sum(1 for v in answers.values() if str(v).strip())
    print(f"  Notebook : {_NB_TITLE}  \u2022  {nd}/{len(answers)} answered")
    raw, src = _grade(answers, _NB_TITLE)
    if raw:
        try:
            result = json.loads(_re.sub(r"```(?:json)?\s*|```","",raw).strip())
        except Exception:
            result = {"quiz_score":nd,"grade":"Good" if nd==len(answers) else "Needs Review","feedback":raw[:400],"tip":""}
    else:
        print("  (Gemini unavailable \u2014 using completion-based feedback)\n")
        result = _fallback(answers)
        src = None
    _show(result, GITHUB_USERNAME.strip(), _NB_TITLE, src)
    print("  \U0001f4be  File \u2192 Save a copy in GitHub \u2192 your fork\n")


---
*Data Pattern Recognition for the Rest of Us · [ladataanalytics.github.io/pattern-recognition-notebooks](https://ladataanalytics.github.io/pattern-recognition-notebooks)*